## SRP512493

**paper:** no official paper but sequencing was likely done in conjuction with the sequencing in this paper [PMID: 39008331](https://pmc.ncbi.nlm.nih.gov/articles/PMC11334205/)

**date, curator:** 2026-01-30, Sara Carsanaro

**notes**
* bears were captured so fully-formed stage is appropriate, it's likely they sequenced adults

### annotation summary

In [19]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,stomach,UBERON:0000945,stomach,perfect match
1,spleen,UBERON:0002106,spleen,perfect match
2,pancreas,UBERON:0001264,pancreas,perfect match
3,ovary,UBERON:0000992,ovary,perfect match
4,lung,UBERON:0002048,lung,perfect match
5,liver,UBERON:0002107,liver,perfect match
6,kidney,UBERON:0002113,kidney,perfect match
7,heart,UBERON:0000948,heart,perfect match


In [20]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,NA,UBERON:0000066,fully formed stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP512493"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 8/8 [00:07<00:00,  1.02it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [6]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24841554,SRP512493,Illumina NovaSeq 6000,SRS21552732,UBERON:0000945,stomach,,,,stomach,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_stomach_246,SAMN40863815,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_stomach_246,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325108,ADC215B20B4116BD0A850A6650FD619D
1,SRX24841553,SRP512493,Illumina NovaSeq 6000,SRS21552731,UBERON:0002106,spleen,,,,spleen,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_spleen_238,SAMN40863814,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_spleen_238,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325109,E10A5D96EFE704183ECF36E0B9434153
2,SRX24841552,SRP512493,Illumina NovaSeq 6000,SRS21552730,UBERON:0001264,pancreas,,,,pancreas,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_pancreas_236,SAMN40863813,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_pancreas_236,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325110,2B09DA1EED09D6AFB123AE565DC7FA56
3,SRX24841551,SRP512493,Illumina NovaSeq 6000,SRS21552729,UBERON:0000992,ovary,,,,ovary,NA,perfect match,not documented,,F,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_ovary_244,SAMN40863812,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_ovary_244,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325111,86A410F4468B60B1801A66F3A4B0D7F8
4,SRX24841550,SRP512493,Illumina NovaSeq 6000,SRS21552728,UBERON:0002048,lung,,,,lung,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_lung_234,SAMN40863811,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_lung_234,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325112,A7356CC3AF9E52E1B68B864965A13ED9
5,SRX24841549,SRP512493,Illumina NovaSeq 6000,SRS21552727,UBERON:0002107,liver,,,,liver,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_liver_232,SAMN40863810,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_liver_232,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325113,E446C63ED29C5562D12E53A2EEAD125A
6,SRX24841548,SRP512493,Illumina NovaSeq 6000,SRS21552726,UBERON:0002113,kidney,,,,kidney,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_kidney_242,SAMN40863809,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_kidney_242,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325114,B1EB42B6C53767F6D6BF97827BFB59EC
7,SRX24841547,SRP512493,Illumina NovaSeq 6000,SRS21552725,UBERON:0000948,heart,,,,heart,NA,perfect match,not documented,,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_heart_240,SAMN40863808,,,,,,,,,,,30/01/20

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['heart' 'kidney' 'liver' 'lung' 'ovary' 'pancreas' 'spleen' 'stomach']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['NA']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24841554,SRP512493,Illumina NovaSeq 6000,SRS21552732,UBERON:0000945,stomach,UBERON:0000066,fully formed stage,,stomach,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_stomach_246,SAMN40863815,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_stomach_246,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325108,ADC215B20B4116BD0A850A6650FD619D
1,SRX24841553,SRP512493,Illumina NovaSeq 6000,SRS21552731,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,spleen,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_spleen_238,SAMN40863814,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_spleen_238,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325109,E10A5D96EFE704183ECF36E0B9434153
2,SRX24841552,SRP512493,Illumina NovaSeq 6000,SRS21552730,UBERON:0001264,pancreas,UBERON:0000066,fully formed stage,,pancreas,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_pancreas_236,SAMN40863813,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_pancreas_236,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325110,2B09DA1EED09D6AFB123AE565DC7FA56
3,SRX24841551,SRP512493,Illumina NovaSeq 6000,SRS21552729,UBERON:0000992,ovary,UBERON:0000066,fully formed stage,,ovary,NA,perfect match,not documented,other,F,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_ovary_244,SAMN40863812,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_ovary_244,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325111,86A410F4468B60B1801A66F3A4B0D7F8
4,SRX24841550,SRP512493,Illumina NovaSeq 6000,SRS21552728,UBERON:0002048,lung,UBERON:0000066,fully formed stage,,lung,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_lung_234,SAMN40863811,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_lung_234,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325112,A7356CC3AF9E52E1B68B864965A13ED9
5,SRX24841549,SRP512493,Illumina NovaSeq 6000,SRS21552727,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_liver_232,SAMN40863810,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_liver_232,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325113,E446C63ED29C5562D12E53A2EEAD125A
6,SRX24841548,SRP512493,Illumina NovaSeq 6000,SRS21552726,UBERON:0002113,kidney,UBERON:0000066,fully formed stage,,kidney,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_kidney_242,SAMN40863809,,,,,,,,,,,30/01/2026,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_kidney_242,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR293251

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-01-30'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24841554,SRP512493,Illumina NovaSeq 6000,SRS21552732,UBERON:0000945,stomach,UBERON:0000066,fully formed stage,,stomach,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_stomach_246,SAMN40863815,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_stomach_246,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325108,ADC215B20B4116BD0A850A6650FD619D
1,SRX24841553,SRP512493,Illumina NovaSeq 6000,SRS21552731,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,spleen,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_spleen_238,SAMN40863814,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_spleen_238,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325109,E10A5D96EFE704183ECF36E0B9434153
2,SRX24841552,SRP512493,Illumina NovaSeq 6000,SRS21552730,UBERON:0001264,pancreas,UBERON:0000066,fully formed stage,,pancreas,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_pancreas_236,SAMN40863813,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_pancreas_236,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325110,2B09DA1EED09D6AFB123AE565DC7FA56
3,SRX24841551,SRP512493,Illumina NovaSeq 6000,SRS21552729,UBERON:0000992,ovary,UBERON:0000066,fully formed stage,,ovary,NA,perfect match,not documented,other,F,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_ovary_244,SAMN40863812,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_ovary_244,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325111,86A410F4468B60B1801A66F3A4B0D7F8
4,SRX24841550,SRP512493,Illumina NovaSeq 6000,SRS21552728,UBERON:0002048,lung,UBERON:0000066,fully formed stage,,lung,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_lung_234,SAMN40863811,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_lung_234,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325112,A7356CC3AF9E52E1B68B864965A13ED9
5,SRX24841549,SRP512493,Illumina NovaSeq 6000,SRS21552727,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_liver_232,SAMN40863810,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_liver_232,,,,not collected,,TRANSCRIPTOMIC,PolyA,SRR29325113,E446C63ED29C5562D12E53A2EEAD125A
6,SRX24841548,SRP512493,Illumina NovaSeq 6000,SRS21552726,UBERON:0002113,kidney,UBERON:0000066,fully formed stage,,kidney,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_kidney_242,SAMN40863809,,,,,,,,,,SAC,2026-01-30,polyA libraries were prepared with KAPA mRNA HyperPrep Kit (Roche) and sequenced with PE100bp reads on a Illumina Novaseq 6000,,Ursame_kidney_242,,,,not collected,,TRANSCRIP

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX24841554,SRP512493,Illumina NovaSeq 6000,SRS21552732,UBERON:0000945,stomach,UBERON:0000066,fully formed stage,,stomach,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_stomach_246,SAMN40863815,,,,,,,,,,SAC,2026-01-30
1,SRX24841553,SRP512493,Illumina NovaSeq 6000,SRS21552731,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,spleen,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_spleen_238,SAMN40863814,,,,,,,,,,SAC,2026-01-30
2,SRX24841552,SRP512493,Illumina NovaSeq 6000,SRS21552730,UBERON:0001264,pancreas,UBERON:0000066,fully formed stage,,pancreas,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_pancreas_236,SAMN40863813,,,,,,,,,,SAC,2026-01-30
3,SRX24841551,SRP512493,Illumina NovaSeq 6000,SRS21552729,UBERON:0000992,ovary,UBERON:0000066,fully formed stage,,ovary,NA,perfect match,not documented,other,F,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_ovary_244,SAMN40863812,,,,,,,,,,SAC,2026-01-30
4,SRX24841550,SRP512493,Illumina NovaSeq 6000,SRS21552728,UBERON:0002048,lung,UBERON:0000066,fully formed stage,,lung,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_lung_234,SAMN40863811,,,,,,,,,,SAC,2026-01-30
5,SRX24841549,SRP512493,Illumina NovaSeq 6000,SRS21552727,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_liver_232,SAMN40863810,,,,,,,,,,SAC,2026-01-30
6,SRX24841548,SRP512493,Illumina NovaSeq 6000,SRS21552726,UBERON:0002113,kidney,UBERON:0000066,fully formed stage,,kidney,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_kidney_242,SAMN40863809,,,,,,,,,,SAC,2026-01-30
7,SRX24841547,SRP512493,Illumina NovaSeq 6000,SRS21552725,UBERON:0000948,heart,UBERON:0000066,fully formed stage,,heart,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_heart_240,SAMN40863808,,,,,,,,,,SAC,2026-01-30


### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP512493,Ursus americanus Transcriptome or Gene expression,"These data have been produced as part of the CCGP. RNA-seq data were generated with Illumina sequencing at the UCLA Technology Center for Genomics and Bioinformatics. For more information, go to ccgproject.org.",SRA,,,,KAPA mRNA HyperPrep Kit,full_length,,PRJNA1011923,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP512493,Ursus americanus Transcriptome or Gene expression,"These data have been produced as part of the CCGP. RNA-seq data were generated with Illumina sequencing at the UCLA Technology Center for Genomics and Bioinformatics. For more information, go to ccgproject.org.",SRA,total,Bgee 1K,8,KAPA mRNA HyperPrep Kit,full_length,,PRJNA1011923,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [15]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39008331'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11334205/'
experiment.loc[:,'DOI'] = '10.1093/jhered/esae037'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP512493,Ursus americanus Transcriptome or Gene expression,"These data have been produced as part of the CCGP. RNA-seq data were generated with Illumina sequencing at the UCLA Technology Center for Genomics and Bioinformatics. For more information, go to ccgproject.org.",SRA,total,Bgee 1K,8,KAPA mRNA HyperPrep Kit,full_length,,PRJNA1011923,39008331,https://pmc.ncbi.nlm.nih.gov/articles/PMC11334205/,10.1093/jhered/esae037,,


#### comments

In [16]:
experiment.loc[:,'comment'] = 'it appears that these libraries were sequenced but not included in this paper, which you can see when you search PRJNA777227 in sra'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP512493,Ursus americanus Transcriptome or Gene expression,"These data have been produced as part of the CCGP. RNA-seq data were generated with Illumina sequencing at the UCLA Technology Center for Genomics and Bioinformatics. For more information, go to ccgproject.org.",SRA,total,Bgee 1K,8,KAPA mRNA HyperPrep Kit,full_length,,PRJNA1011923,39008331,https://pmc.ncbi.nlm.nih.gov/articles/PMC11334205/,10.1093/jhered/esae037,,"it appears that these libraries were sequenced but not included in this paper, which you can see when you search PRJNA777227 in sra"


#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [21]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [22]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
53412,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-047,SAMN18642346,,,,,,,iliac crest (UBERON:0014437) may also be appro...,hibernating,,SAC,2026-01-30
53413,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-056,SAMN18642345,,,,,,,iliac crest (UBERON:0014437) may also be appro...,hibernating,,SAC,2026-01-30
53414,SRX24841554,SRP512493,Illumina NovaSeq 6000,SRS21552732,UBERON:0000945,stomach,UBERON:0000066,fully formed stage,,stomach,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_stomach_246,SAMN40863815,,,,,,,,,,SAC,2026-01-30
53415,SRX24841553,SRP512493,Illumina NovaSeq 6000,SRS21552731,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,spleen,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_spleen_238,SAMN40863814,,,,,,,,,,SAC,2026-01-30
53416,SRX24841552,SRP512493,Illumina NovaSeq 6000,SRS21552730,UBERON:0001264,pancreas,UBERON:0000066,fully formed stage,,pancreas,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_pancreas_236,SAMN40863813,,,,,,,,,,SAC,2026-01-30
53417,SRX24841551,SRP512493,Illumina NovaSeq 6000,SRS21552729,UBERON:0000992,ovary,UBERON:0000066,fully formed stage,,ovary,NA,perfect match,not documented,other,F,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_ovary_244,SAMN40863812,,,,,,,,,,SAC,2026-01-30
53418,SRX24841550,SRP512493,Illumina NovaSeq 6000,SRS21552728,UBERON:0002048,lung,UBERON:0000066,fully formed stage,,lung,NA,perfect match,not documented,other,NA,,,9643,KAPA mRNA HyperPrep Kit,full_length,polyA,,,Ursame_lung_234,SAMN40863811,,,,,,,,,,SAC,2026-01-30


In [23]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1019,SRP075217,"Whole-genome sequencing, de novo assembly, and...",we have performed high-throughput sequencing a...,SRA,partial,Bgee 1K,12,TruSeq Stranded Total RNA,full_length,,PRJNA319796,30395234,https://pmc.ncbi.nlm.nih.gov/articles/PMC6379037/,10.1093/dnares/dsy036,11861896[PMID],removed DNA libraries
1020,SRP313712,Transcriptional changes and preservation of bo...,Physical inactivity leads to losses of bone ma...,SRA,total,Bgee 1K,8,,,,PRJNA720155,33859306,https://pmc.ncbi.nlm.nih.gov/articles/PMC8050052/,10.1038/s41598-021-87785-9,,
1021,SRP512493,Ursus americanus Transcriptome or Gene expression,These data have been produced as part of the C...,SRA,total,Bgee 1K,8,KAPA mRNA HyperPrep Kit,full_length,,PRJNA1011923,39008331,https://pmc.ncbi.nlm.nih.gov/articles/PMC11334...,10.1093/jhered/esae037,,it appears that these libraries were sequenced...


### add annotations to git

In [24]:
! git pull

Already up to date.


In [25]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [26]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [27]:
! git add $git_experiment_path $git_library_path

In [28]:
! git commit -m $commit_message_exp

[develop afa6d7d] adding annotated bulk experiment SRP512493
 2 files changed, 9 insertions(+)


In [29]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.07 KiB | 2.07 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   0491672..afa6d7d  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push